# Symbolic Module Showcase

This notebook demonstrates the capabilities of the `symbolic` module in `conservative_pid`. 
It explains the different modalities (Variables, Counterfactual Terms, Events, Queries) and investigates how to handle complex queries involving logical and arithmetic operations.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Add parent directory to path to allow importing local modules
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from symbolic import Variable, CounterfactualTerm, Event, P, Query, Expression
from inference import ConservativePID

## 1. Setup Environment
We use the same 2-variable binary setup (X->Y) as in the binary experiment to illustrate concepts.

In [2]:
# Define Variables
X = Variable("X", (0, 1))
Y = Variable("Y", (0, 1))
variables = [X, Y]

# Mock Data (Relative Frequencies computed previously)
observational_data = {(0, 0): 0.35, (0, 1): 0.153, (1, 0): 0.096, (1, 1): 0.401}

# Initialize Solver
pid = ConservativePID(variables, observational_data)

## 2. The Symbolic Building Blocks

The `symbolic` module provides a hierarchy of classes to represent causal queries.

### A. Variable
The atomic unit. Represents a random variable with a finite domain.

In [3]:
print(f"Variable: {X}, Domain: {X.domain}")

Variable: X, Domain: (0, 1)


### B. CounterfactualTerm
Represents a variable under an intervention, e.g., $Y_{X=1}$.
We use the `@` operator syntax sugar: `Variable @ InterventionDict`.

In [4]:
Y_x1 = Y @ {X: 1}
print(f"Counterfactual Term: {Y_x1}")

# Nested counterfactuals are also supported (though less common in simple DAGs)
W = Variable("W", (0, 1))
nested = Y @ {X: 1, W @ {X: 0}: 1}
# print(f"Nested Term: {nested}")

Counterfactual Term: Y_{X=1}


### C. Event
An `Event` is a **conjunction** of atomic propositions (assignments). 
It represents a state of the world where specific counterfactual terms take specific values.

Syntax sugar:
- `Y_x1 == 1` produces an `Event` with a single assignment.
- `&` combines events (Conjunction). 

In [5]:
# Atomic Event
event1 = Y_x1 == 1
print(f"Atomic Event: {event1}")

# Conjunction
event2 = X == 0
joint_event = event1 & event2
print(f"Joint Event: {joint_event}")

# Note: Dictionary representation inside Event
print(f"Internal Assignments: {joint_event.assignments}")

Atomic Event: Y_{X=1} = 1
Joint Event: Y_{X=1} = 1 & X = 0
Internal Assignments: {Y_{X=1}: 1, X: 0}


### D. Query
A `Query` represents a probability expression $P(\text{target} \mid \text{evidence})$.
Use the `P()` helper function.

In [6]:
# Simple Probability P(Y_{x=1} = 1)
q1 = P(Y_x1 == 1)
print(f"Query 1: {q1}")

# Conditional Probability P(Y_{x=1} = 1 | X = 0)
q2 = P(Y_x1 == 1, X == 0)
print(f"Query 2: {q2}")

Query 1: P(Y_{X=1} = 1)
Query 2: P(Y_{X=1} = 1 | X = 0)


### E. Expressions

We can now form linear combinations of queries (Expressions) using standard arithmetic operators `+`, `-`, `*`.
This is useful for defining quantities like the Average Treatment Effect (ATE) or Causal Risk Differences.

In [14]:
# 1. Addition of Queries
expr_sum = P(X == 1) + P(Y == 1)
assert isinstance(expr_sum, Expression)
print(f"Sum Expression: {expr_sum}")

# 2. Subtraction (Difference)
expr_diff = P(Y_x1 == 1) - P(Y @ {X: 0} == 1)
assert isinstance(expr_diff, Expression)
print(f"Difference Expression (ATE-like): {expr_diff}")

# 3. Scalar Multiplication
expr_scaled = 2 * P(X == 1) - 0.5 * P(Y == 1)
assert isinstance(expr_scaled, Expression)
print(f"Scaled Expression: {expr_scaled}")

# 4. Negation
expr_neg = -P(X == 1)
assert isinstance(expr_neg, Expression)
print(f"Negated Query: {expr_neg}")

Sum Expression: P(X = 1) + P(Y = 1)
Difference Expression (ATE-like): P(Y_{X=1} = 1) - P(Y_{X=0} = 1)
Scaled Expression: 2.0 * P(X = 1) - 0.5 * P(Y = 1)
Negated Query: - P(X = 1)
